# 0. Qué datos tenemos para hacer machine learning

Antes de entrenar nada conviene saber **qué hay en el almacén y qué se puede predecir con ello**. Este cuaderno no entrena modelos: es un mapa.

## Recordatorio de las capas 

| Capa | Qué es | ¿Se usa para ML? |
|---|---|---|
| **Datos recibidos (bronce)** | Los ficheros tal como los exporta el sistema, con sus errores | No |
| **Plata** | Los mismos datos limpios: un fichero por listado, con cada corrección registrada | No directamente |
| **Oro** | El almacén `almacen.duckdb`: los datos de plata organizados en **hechos** y **dimensiones** (modelo en estrella) | **Sí: todo el ML sale de aquí** |

En oro hay dos tipos de tabla:

- **Hechos (`fact_…`)**: lo que ha pasado. Cada fila es un evento con números: una línea vendida, una proforma, un cobro.
- **Dimensiones (`dim_…`)**: el quién, qué y cuándo de cada hecho: cliente, producto, vendedor, fecha. Los hechos guardan claves (`cliente_key`, `producto_key`…) que apuntan a ellas.

**Para hacer ML se cruza un hecho con sus dimensiones y se agrega al nivel de lo que queremos predecir** (un mes, una proforma, una factura, un cliente).


In [ ]:
# Instalar una sola vez (en Anaconda Prompt o en una celda con !):
%pip install duckdb pandas scikit-learn statsmodels mlxtend matplotlib pyarrow

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

from comun import *   # conexion al almacen (con, q), CORTE, guardar(), estilo(), mape(), colores

## 1. Qué tablas hay y cuántas filas tienen

In [ ]:
q('''
    SELECT table_name AS tabla, estimated_size AS filas FROM duckdb_tables()
    WHERE table_name LIKE 'fact_%' OR table_name LIKE 'dim_%'
    ORDER BY tabla
''')

## 2. Qué significa cada columna

El diccionario de datos lo explica columna a columna. Cambia el nombre de la tabla para ver otra.

In [ ]:
diccionario = pd.read_csv('../oro/diccionario_datos.csv')
pd.set_option('display.max_colwidth', 100)
diccionario.query("TABLA == 'fact_ventas'")[['COLUMNA', 'DESCRIPCION']]

## 3. Qué se puede predecir: las cinco preguntas

Con estas tablas hay cinco preguntas de negocio que se pueden responder con ML. Cada una tiene su cuaderno:

| Cuaderno | Pregunta | Tipo de problema | Una fila = | Qué se predice (variable objetivo) | Tabla de partida |
|---|---|---|---|---|---|
| 01 | ¿Cuánto venderemos los próximos 12 meses? | Serie temporal | un mes | ventas netas del mes | `v_ventas_mensuales` |
| 02 | ¿Qué proformas acabarán en venta? | Clasificación | una proforma | `convertida` (1/0) | `fact_embudo_proforma` |
| 03 | ¿Qué facturas a crédito se pagarán tarde? | Clasificación | una factura a crédito | pago con más de 15 días de retraso (1/0) | `fact_ventas` + `fact_cobros` |
| 04 | ¿Qué tipos de cliente tenemos? | No supervisado (agrupación) | un cliente | (no hay objetivo: se buscan grupos) | `fact_ventas` |
| 05 | ¿Qué familias se compran juntas? | Reglas de asociación | un comprobante | (no hay objetivo: se buscan patrones) | `fact_ventas` |

Las cinco cubren lo que promete el OE5 de la memoria: previsión de ventas, embudo proforma–factura, segmentación de clientes y análisis de producto, cruzando ventas y cobros.

**¿Qué no se puede predecir?** Inventario, compras y roturas de stock: no hay datos (decisión del Bloque 1, Tabla 11 como ampliación futura).

## 4. Un vistazo a cada fuente

### Ventas por mes (cuaderno 01)
32 meses, de enero de 2024 a agosto de 2026. Es poco para una serie temporal: menos de tres ciclos anuales.

In [ ]:
vm = q('SELECT anio_mes, ventas_netas_pen, margen_pct, n_comprobantes, n_clientes FROM v_ventas_mensuales ORDER BY anio_mes')
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(pd.to_datetime(vm.anio_mes), vm.ventas_netas_pen / 1000, color=AZUL, lw=2)
ax.set_ylim(bottom=0)
estilo(ax, 'Ventas netas por mes', 'miles de S/')
plt.show()
vm.tail()

### Proformas (cuaderno 02)
Cada proforma acaba **cerrada** (se vendió), **vencida** o **rechazada**; las **abiertas** aún no se sabe.

In [ ]:
q('''
    SELECT ep.estado, COUNT(*) AS proformas, ROUND(AVG(e.convertida) * 100, 1) AS pct_convertidas
    FROM fact_embudo_proforma e JOIN dim_estado_proforma ep USING (estado_proforma_key)
    GROUP BY 1 ORDER BY 2 DESC
''')

### Cobros (cuaderno 03)
`dias_sobre_vencimiento` dice cuántos días después del vencimiento se cobró (negativo = antes).

In [ ]:
cob = q('''
    SELECT c.dias_sobre_vencimiento FROM fact_cobros c
    JOIN dim_condicion_pago cp USING (condicion_pago_key)
    WHERE c.tipo_movimiento = 'COBRO' AND cp.es_credito
''')
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(cob.dias_sobre_vencimiento.clip(-20, 120), bins=70, color=AZUL)
ax.axvline(15, color=ROJO, ls='--'); ax.text(17, ax.get_ylim()[1] * .85, 'más de 15 días = pago tardío', color=ROJO, fontsize=9)
estilo(ax, 'Días de retraso sobre el vencimiento (ventas a crédito)', 'cobros')
plt.show()

### Clientes (cuaderno 04)

In [ ]:
q('''
    SELECT tipo_cliente, COUNT(*) AS clientes FROM dim_cliente
    WHERE es_actual AND cliente_id <> '00000000' GROUP BY 1 ORDER BY 2 DESC
''')

## 5. Tres reglas que se respetan en todos los cuadernos

1. **Importes en soles y sin IGV** (columnas `*_pen`). Las notas de crédito ya restan.
2. **No mirar al futuro (sin fuga de información).** Para predecir algo solo se usan variables que se conocían en ese momento. Por ejemplo, para predecir si una proforma se venderá no vale usar el importe facturado, porque ese dato solo existe si se vendió.
3. **Validación temporal.** Se entrena con el pasado y se evalúa con un periodo posterior que el modelo no ha visto, igual que se usaría en la empresa. Nunca se mezclan al azar meses de 2026 en el entrenamiento.

**Para la memoria:** las cifras describen el periodo analizado (enero de 2024 a agosto de 2026). Lo que se lleva a la empresa sin cambios es el método: cómo se construyen las variables, cómo se valida cada modelo y cómo se reentrena con datos nuevos.